In [0]:
%sql
use catalog ext_cat;

In [0]:
landing_zone =  '/Volumes/ext_cat/default/raw'
orders_data = landing_zone + '/ordershistory'
checkpoint_path =  landing_zone + '/orders_checkpoint'

Define the schema in advance

In [0]:
from pyspark.sql.types import *
orderSchema = StructType ([
    StructField("order_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_status", StringType(), True),    
])
    #StructField("_rescued_data", StringType(), True)

In [0]:
ordersdf= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", checkpoint_path) \
  .option('header','true') \
  .schema(orderSchema) \
  .option('rescuedDataColumn', '_rescued_data') \
  .load(orders_data)
  # .option("cloudFiles.inferSchema", "true") \
  # .option("cloudFiles.inferColumnTypes", 'true') \

In [0]:
%sql
drop table if exists ext_cat.default.orders;

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from ext_cat.default.orders;

order_id,order_date,customer_id,order_status,_rescued_data
1001,2026-08-25,201,Delivered,null
1002,2026-08-26,205,Shipped,null
1003,2026-08-27,202,Processing,null
1004,2026-08-28,208,Cancelled,null
1005,2026-08-30,201,Delivered,null
1006,2026-09-01,210,Shipped,null
1007,2026-09-02,204,Processing,null
1008,2026-09-03,207,Pending,null
1009,2026-09-05,203,Pending,null


Add new file and rerun

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from ext_cat.default.orders;

order_id,order_date,customer_id,order_status,_rescued_data
10101,null,2101,Delivered,"{""order_date"":""25-08-2025"",""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10102,null,2105,Shipped,"{""order_date"":""26-08-2025"",""new_col"":""2"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10103,null,2102,Processing,"{""order_date"":""27-08-2025"",""new_col"":""3"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10104,null,2108,Cancelled,"{""order_date"":""28-08-2025"",""new_col"":""4"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10105,null,2101,Delivered,"{""order_date"":""30-08-2025"",""new_col"":""5"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10106,null,2110,Shipped,"{""order_date"":""01-09-2025"",""new_col"":""6"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10107,null,2104,Processing,"{""order_date"":""02-09-2025"",""new_col"":""7"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10108,null,2107,Pending,"{""order_date"":""03-09-2025"",""new_col"":""8"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
10109,null,1203,Pending,"{""order_date"":""05-09-2025"",""new_col"":""1"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}"
1001,2026-08-25,201,Delivered,null


Since there is no schema evolution and infer schema, new colum and data type mismatches in new files goes to rescued col, even with mergschema in writestream

Adding **Schema Hint** and metadata cols

In [0]:
%sql
drop table if exists ext_cat.default.orders;

In [0]:
from pyspark.sql.functions import *
ordersdf= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", checkpoint_path) \
  .option("cloudFiles.inferSchema", "true") \
  .option("cloudFiles.schemaHints", "order_id int, order_date date, customer_id int, order_status string") \
  .option("cloudFiles.inferColumnTypes", 'true') \
  .load(orders_data) \
  .withColumn("file_name",col("_metadata.file_name")) \
  .withColumn("ingesttime",current_timestamp())
  

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from ext_cat.default.orders;

order_id,order_date,customer_id,order_status,_rescued_data,file_name,ingesttime
1001,2026-08-25,201,Delivered,null,orders1.csv,2026-09-05T18:34:10.369Z
1002,2026-08-26,205,Shipped,null,orders1.csv,2026-09-05T18:34:10.369Z
1003,2026-08-27,202,Processing,null,orders1.csv,2026-09-05T18:34:10.369Z
1004,2026-08-28,208,Cancelled,null,orders1.csv,2026-09-05T18:34:10.369Z
1005,2026-08-30,201,Delivered,null,orders1.csv,2026-09-05T18:34:10.369Z
1006,2026-09-01,210,Shipped,null,orders1.csv,2026-09-05T18:34:10.369Z
1007,2026-09-02,204,Processing,null,orders1.csv,2026-09-05T18:34:10.369Z
1008,2026-09-03,207,Pending,null,orders1.csv,2026-09-05T18:34:10.369Z
1009,2026-09-05,203,Pending,null,orders1.csv,2026-09-05T18:34:10.369Z


Add a new file with a new col

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

new cols will get loaded after a retry of read and re run of write

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from ext_cat.default.orders;

order_id,order_date,customer_id,order_status,_rescued_data,file_name,ingesttime,new_col
10101,2025-08-25,2101,Delivered,null,orders3.csv,2026-09-05T18:39:50.333Z,1
10102,2025-08-26,2105,Shipped,null,orders3.csv,2026-09-05T18:39:50.333Z,1
10103,2025-08-27,2102,Processing,null,orders3.csv,2026-09-05T18:39:50.333Z,1
10104,2025-08-28,2108,Cancelled,null,orders3.csv,2026-09-05T18:39:50.333Z,1
10105,2025-08-30,2101,Delivered,null,orders3.csv,2026-09-05T18:39:50.333Z,1
10106,2025-09-01,2110,Shipped,null,orders3.csv,2026-09-05T18:39:50.333Z,1
10107,2025-09-02,2104,Processing,null,orders3.csv,2026-09-05T18:39:50.333Z,1
10108,2025-09-03,2107,Pending,null,orders3.csv,2026-09-05T18:39:50.333Z,1
10109,2025-09-05,1203,Pending,null,orders3.csv,2026-09-05T18:39:50.333Z,1
1001,2026-08-25,201,Delivered,null,orders1.csv,2026-09-05T18:34:10.369Z,null
